# Type I and Type II Errors — and the Significance Level $\alpha$

Perfect decisions are impossible: the world is random and you hold only partial information. So a hypothesis test can fail in exactly two ways, and both get names. Send a good email to the spam box — reject $H_0$ when it was true — and you've made a **Type I error** (a *false positive*). Let a spam email into the inbox — fail to reject $H_0$ when it was false — and that's a **Type II error** (a *false negative*). You can never know which cell of the outcome table you landed in; what you *can* do is design the test so the error **rates** stay below an accepted threshold. The maximum Type I probability you tolerate is the **significance level $\alpha$** — typically 0.05 — and choosing it is choosing your decision threshold. It is based on the second video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

The last notebook ended with two unnamed mistakes lurking in the toy detector. This one names them, tabulates them, prices them, and introduces the dial that controls the first of them — the same Greek letter $\alpha$ that ran the confidence-interval lesson, now wearing its original hat.

After completing this notebook you will be able to:

- Define Type I (false positive) and Type II (false negative) errors and place them in the 2×2 outcome table.
- Explain why you never know whether any single decision was right — and why error *rates* are the design target instead.
- Argue which error is more severe in a given problem, and how that judgment is already baked into the choice of $H_0$.
- Define the significance level $\alpha$ as the maximum tolerated Type I probability, and dismantle the extremes $\alpha = 0$ and $\alpha = 1$.
- Demonstrate the catch: at a fixed sample size, pushing Type I errors down pushes Type II errors up.

## 2. Theory

### What could go wrong

The test has two possible outputs — spam box or inbox — and each can be wrong:

- **Type I error (false positive):** you **reject $H_0$ when it was actually true.** A perfectly good ham email lands in the spam box.
- **Type II error (false negative):** you **fail to reject $H_0$ when it was actually false.** A spam email strolls into the inbox.

### The outcome table

| | decision: don't reject $H_0$ | decision: reject $H_0$ |
|---|---|---|
| **$H_0$ true** (email is ham) | ✅ correct — good email delivered | ❌ **Type I** — good email lost to spam |
| **$H_1$ true** (email is spam) | ❌ **Type II** — spam in the inbox | ✅ correct — spam caught |

A sobering feature: **you never know which cell you're in**, because the ground truth (the row) is exactly what you don't observe. All you can control is the long-run rate at which each error happens — so that becomes the design goal.

### Not all errors cost the same

For email, losing a good message (Type I) is far worse than reading the occasional spam (Type II) — you'd much rather find an accidental spam in the inbox than never see a good email again. So here **Type I errors are the severe ones**. This is no coincidence: back in notebook 01, the safe claim was *made* $H_0$ precisely so that wrongly rejecting it would be the expensive mistake. Choosing $H_0$ and deciding which error is scary are two views of one design decision.

### The significance level $\alpha$

How much Type I risk will you tolerate? On average, how many ham emails may go to spam so that the detector still catches most spam? That negotiated ceiling is the **significance level**:

$$
\alpha = \text{maximum probability of a Type I error} = \max P(\text{reject } H_0 \mid H_0 \text{ true})
$$

It's a probability, so $0 \le \alpha \le 1$ — and both endpoints are terrible detectors:

- **$\alpha = 0$:** never reject, whatever the evidence. Every email is "ham," no Type I errors ever — and every spam sails through.
- **$\alpha = 1$:** always reject. Every email is "spam," and every ham email becomes a Type I error.

Useful tests live in between; **0.05** is the typical choice (on average, 1 in 20 ham emails misjudged), with **0.01** the common stricter option.

### The catch

You'd like $\alpha$ as small as possible — but at a **fixed sample size**, squeezing the Type I probability raises the Type II probability. (That is exactly the $\alpha = 0$ scenario taken to its limit: zero false positives, maximal false negatives.) The tension is fundamental, and the only clean escape — more data — comes later in the lesson.

$\alpha$ is your **design criterion**: fix it first, and it determines the threshold your sample must cross for $H_0$ to be rejected.

## 3. Worked Examples

### Example 1: the table, filled in with consequences

Ham email, kept: ✅ (the everyday case). Ham email, flagged: ❌ Type I — the irreplaceable loss. Spam, flagged: ✅ (the detector doing its job). Spam, kept: ❌ Type II — a nuisance, deleted by hand.

### Example 2: same errors, other courtrooms

| problem | $H_0$ | Type I means... | Type II means... | severe one |
|---|---|---|---|---|
| criminal trial | innocent | convicting an innocent | acquitting the guilty | Type I |
| smoke alarm | no fire | false alarm | sleeping through a fire | Type II |
| medical screening | healthy | false scare, more tests | missed disease | usually Type II |

Note the smoke alarm: society happily accepts a huge Type I rate (burnt toast) to keep Type II near zero. "Which error is worse" is not fixed by mathematics — it's the engineering judgment that decided $H_0$, revisited.

### Example 3: from $\alpha$ to a threshold

Notebook 01's detector counts trigger phrases; each appears in ham with probability 0.001 (6 phrases). Rule: reject at $\ge k$. The Type I probability of the rule is $P(\text{count} \ge k \mid \text{ham})$:

- $k = 1$: $P \approx 0.006$ — already below $\alpha = 0.05$, so $\alpha = 0.05$ licenses the aggressive rule $k = 1$.
- $k = 2$: $P \approx 1.5 \times 10^{-5}$ — needed once you demand $\alpha = 10^{-3}$.
- $k = 3$: $P \approx 2 \times 10^{-8}$ — the rule $\alpha = 10^{-6}$ forces.

Fix $\alpha$, and the smallest $k$ whose ham-tail fits under it *is* your threshold: the design criterion in action.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from math import comb

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(6)

def z_score(p):
    """Normal critical value z_p (probability p to the left)."""
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

Phi = lambda z: 0.5 * (1 + float(sp.erf(sp.Float(z) / sp.sqrt(2))))

### The outcome table, filled by simulation

100,000 emails through the $k \ge 2$ detector from notebook 01, sorted into the four cells — with the two error rates read off the margins.

In [ ]:
P_HAM_TRIG, P_SPAM_TRIG, N_PHRASES = 0.001, 0.40, 6

n_emails = 100_000
is_spam = rng.random(n_emails) < 0.05
counts = np.where(is_spam,
                  rng.binomial(N_PHRASES, P_SPAM_TRIG, n_emails),
                  rng.binomial(N_PHRASES, P_HAM_TRIG, n_emails))
reject = counts >= 2

cells = {"ham, kept  (correct)": (~is_spam & ~reject).sum(),
         "ham, flagged  (TYPE I)": (~is_spam & reject).sum(),
         "spam, flagged (correct)": (is_spam & reject).sum(),
         "spam, kept   (TYPE II)": (is_spam & ~reject).sum()}
for name, count in cells.items():
    print(f"{name:>24s}: {count:>6d}")

type1_rate = (~is_spam & reject).sum() / (~is_spam).sum()
type2_rate = (is_spam & ~reject).sum() / is_spam.sum()
print(f"\nrealized Type I rate  P(reject | ham)  = {type1_rate:.4%}")
print(f"realized Type II rate P(keep | spam)   = {type2_rate:.2%}")
print("(Which individual emails were misjudged? Unknowable without ground truth --")
print(" only these long-run rates were ever under our control.)")

### $\alpha$ picks the threshold

For each candidate rule $k$, the Type I probability is the ham-tail $P(\text{count} \ge k \mid \text{ham})$. Given a demanded $\alpha$, the threshold is the smallest $k$ that fits under it.

In [ ]:
def ham_tail(k):
    return sum(comb(N_PHRASES, j) * P_HAM_TRIG**j * (1 - P_HAM_TRIG)**(N_PHRASES - j)
               for j in range(k, N_PHRASES + 1))

print("rule      P(Type I) = P(count >= k | ham)")
for k in range(1, 5):
    print(f"k >= {k}:   {ham_tail(k):.2e}")

for alpha in (0.05, 1e-3, 1e-6):
    k_alpha = next(k for k in range(0, N_PHRASES + 2) if ham_tail(k) <= alpha)
    print(f"\nalpha = {alpha:g} -> threshold k = {k_alpha} "
          f"(Type I = {ham_tail(k_alpha):.2e} <= alpha)")

## 5. Visualizations

### The classic picture: $\alpha$ and the Type II region share one threshold

A generic test where some evidence score is $N(0,1)$ under $H_0$ and $N(3,1)$ under $H_1$. The threshold set by $\alpha$ splits *both* curves: the $H_0$ tail beyond it is the Type I probability, the $H_1$ mass before it is the Type II probability. Slide the threshold and watch the see-saw.

In [ ]:
score = np.linspace(-4, 7, 700)
f0 = np.exp(-0.5 * score**2) / np.sqrt(2 * np.pi)              # under H0
f1 = np.exp(-0.5 * (score - 3)**2) / np.sqrt(2 * np.pi)        # under H1

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
for ax, alpha in zip(axes, (0.20, 0.05, 0.001)):
    thr = z_score(1 - alpha)
    beta = Phi(thr - 3)
    ax.plot(score, f0, color="tab:green", linewidth=1.8, label="under $H_0$")
    ax.plot(score, f1, color="tab:red", linewidth=1.8, label="under $H_1$")
    ax.fill_between(score, f0, where=score >= thr, color="tab:green", alpha=0.45)
    ax.fill_between(score, f1, where=score <= thr, color="tab:red", alpha=0.35)
    ax.axvline(thr, color="black", linestyle="--", linewidth=1.3)
    ax.set_title(f"$\\alpha$ = {alpha:g} -> threshold {thr:.2f}\n"
                 f"Type II prob = {beta:.1%}", fontsize=10)
    ax.set_xlabel("evidence score")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=9)
plt.suptitle("Shrinking the green tail (Type I) grows the red belly (Type II): the see-saw")
plt.tight_layout()
plt.show()

### The see-saw as a curve

Every choice of $\alpha$ buys a Type II probability. The whole menu at once — for this fixed evidence quality, there is no point with both errors tiny.

In [ ]:
alphas = np.logspace(-4, np.log10(0.5), 200)
betas = [Phi(z_score(1 - a) - 3) for a in alphas]

plt.figure(figsize=(9, 4.6))
plt.plot(alphas, betas, color="tab:purple", linewidth=2)
for a in (0.05, 0.01, 0.001):
    b = Phi(z_score(1 - a) - 3)
    plt.plot(a, b, "ko", markersize=5)
    plt.annotate(f"$\\alpha$ = {a:g}: Type II = {b:.1%}", xy=(a, b),
                 xytext=(8, 6), textcoords="offset points", fontsize=9)
plt.xscale("log")
plt.xlabel(r"significance level $\alpha$ (Type I ceiling, log scale)")
plt.ylabel("Type II probability")
plt.title("Fixed sample, fixed evidence quality: lowering one error raises the other")
plt.grid(True, alpha=0.3)
plt.show()

### The whole story on one page

Everything above in a single sheet: the four outcomes for 100,000 emails, the two error curves cut by one threshold, the Type I / Type II see-saw, and how α picks the rule.

<img src="img/02_type_errors.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Classify each mistake as Type I or Type II (state $H_0$ first): (a) a fire alarm stays silent during a real fire; (b) a fair coin is declared biased; (c) a fraudulent transaction is approved; (d) an innocent defendant is convicted.

**Exercise 2.** Describe the $\alpha = 0$ and $\alpha = 1$ spam detectors in one sentence each: what do they do with every email, which error do they eliminate, and which do they maximize?

### Intermediate

**Exercise 3.** A stricter trigger-phrase world has $P(\text{phrase} \mid \text{ham}) = 0.01$ (still 6 phrases). Recompute the threshold $k$ demanded by $\alpha = 0.05$, $0.01$, and $0.001$, and the Type II rate each threshold implies (spam still triggers each phrase with probability 0.4).

**Exercise 4.** In the courtroom table, "beyond reasonable doubt" plays the role of a very small $\alpha$. Explain what the legal system implicitly says about the two error costs, and which quantity grows as a consequence (connect to the see-saw).

### Advanced

**Exercise 5.** *The see-saw, escaped.* In the two-Gaussian world, averaging $n$ independent evidence scores turns the separation from 3 into $3\sqrt{n}$ (standard deviations stay 1 after rescaling). At fixed $\alpha = 0.01$, compute and plot the Type II probability against $n = 1, \dots, 6$. How many scores are needed for Type II $\le 1\%$ — and what does this preview about the role of sample size?

**Exercise 6.** *Auditing a claimed $\alpha$.* A vendor claims their detector has $\alpha = 0.05$. You feed it 10,000 known-ham emails and it flags 561. Using the sampling machinery of Lesson 7 (a proportion confidence interval!), decide whether the claim is credible — and notice which lesson's tool just audited which lesson's parameter.

## 7. Solutions

In [ ]:
# Exercise 1: classifications
print("(a) H0: no fire. Silent during a fire = fail to reject a false H0 -> TYPE II.")
print("(b) H0: coin fair. Declared biased when fair = wrongly rejected H0 -> TYPE I.")
print("(c) H0: transaction legitimate. Fraud approved = kept a false H0 -> TYPE II.")
print("(d) H0: innocent. Convicted innocent = wrongly rejected H0 -> TYPE I.")

In [ ]:
# Exercise 2: the two extreme detectors
print("alpha = 0: rejects nothing -- every email is 'ham'; Type I eliminated,")
print("           Type II maximized (all spam delivered).")
print("alpha = 1: rejects everything -- every email is 'spam'; Type II eliminated,")
print("           Type I maximized (every ham email lost).")

In [ ]:
# Exercise 3: noisier ham, new thresholds
p_ham3 = 0.01
def ham_tail3(k):
    return sum(comb(6, j) * p_ham3**j * (1 - p_ham3)**(6 - j) for j in range(k, 7))
def spam_below(k):
    return sum(comb(6, j) * 0.4**j * 0.6**(6 - j) for j in range(0, k))

for alpha in (0.05, 0.01, 0.001):
    k_a = next(k for k in range(0, 8) if ham_tail3(k) <= alpha)
    print(f"alpha = {alpha:g}: threshold k = {k_a}  "
          f"(Type I = {ham_tail3(k_a):.2e}, Type II = {spam_below(k_a):.1%})")
print("\nNoisier ham pushes the thresholds up -- and the Type II bill grows with them.")

In [ ]:
# Exercise 4: the courtroom's alpha
print("'Beyond reasonable doubt' = demand overwhelming evidence before rejecting")
print("H0 (innocence): a tiny alpha. The system thereby declares convicting the")
print("innocent (Type I) vastly costlier than acquitting the guilty (Type II) --")
print("and, by the see-saw, knowingly accepts MORE guilty verdicts of 'not guilty'")
print("(a higher Type II rate) as the price of that protection.")

In [ ]:
# Exercise 5: sample size breaks the see-saw
alpha5 = 0.01
thr5 = z_score(1 - alpha5)
ns = np.arange(1, 7)
betas5 = [Phi(thr5 - 3 * np.sqrt(n)) for n in ns]

plt.figure(figsize=(8.5, 4.2))
plt.plot(ns, betas5, "o-", color="tab:blue", linewidth=2)
plt.axhline(0.01, color="black", linestyle=":", label="Type II target 1%")
plt.xlabel("independent evidence scores averaged (n)")
plt.ylabel(r"Type II probability at fixed $\alpha$ = 0.01")
plt.title("Exercise 5: more data lowers Type II without touching Type I")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

for n, b in zip(ns, betas5):
    print(f"n = {n}: Type II = {b:.2%}")
n_star = next(int(n) for n, b in zip(ns, betas5) if b <= 0.01)
print(f"\nType II <= 1% from n = {n_star} on: sample size is the see-saw's escape")
print("hatch -- the preview of power analysis later in the lesson.")

In [ ]:
# Exercise 6: auditing the vendor with Lesson 7 tools
flagged, n_audit = 561, 10_000
p_hat = flagged / n_audit
se = np.sqrt(p_hat * (1 - p_hat) / n_audit)
z95 = z_score(0.975)
lo, hi = p_hat - z95 * se, p_hat + z95 * se
print(f"observed Type I rate: {p_hat:.2%}")
print(f"95% confidence interval: [{lo:.4f}, {hi:.4f}]")
print(f"claimed alpha = 0.05 inside the interval? {lo <= 0.05 <= hi}")
print("\nThe interval excludes 0.05 -- the audit contradicts the claim: this")
print("detector's true false-positive rate is credibly above 5%. (Lesson 7's")
print("proportion interval just audited Lesson 8's significance level.)")

## 8. Conclusion

- Two errors, two names: **Type I / false positive** (reject a true $H_0$ — ham to the spam box) and **Type II / false negative** (keep a false $H_0$ — spam in the inbox); the 2×2 table holds all four outcomes.
- The ground truth is never revealed, so single decisions can't be graded — tests are designed around long-run **error rates**.
- Error costs are asymmetric by design: $H_0$ was chosen so that Type I is the expensive mistake (the courtroom's convicted innocent; but note the smoke alarm, where the roles flip).
- The **significance level $\alpha$** is the negotiated ceiling on Type I probability — 0.05 typically, 0.01 when stricter; $\alpha = 0$ and $\alpha = 1$ are both useless extremes — and fixing $\alpha$ fixes the rejection threshold.
- The catch: at fixed sample size, less Type I means more Type II (the see-saw) — with more data as the escape hatch, previewed in the exercises.

Next: turning a sample into the actual test statistic that gets compared against $\alpha$'s threshold.